# 📁 Chapter 2: Automating Files and the Filesystem

📖 Book: Chapter 2, pp. 82–111 · ⏱️ 2–3 hours · 🧱 Needs: Ch 1 · Level: Beginner

---

> **🎯 Key idea:** Using `with open(..., encoding='utf-8')`, `pathlib.Path`, and streaming generators eliminates file-descriptor leaks and cross-OS path bugs.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `os.path.join(a, b)`  
> **🆕 Now (2026):** `pathlib.Path(a) / b`  
> **Why:** Object-oriented, cross-OS `/` operator

> **📜 Then (book, ~2020):** `yaml.load(f)`  
> **🆕 Now (2026):** `yaml.safe_load(f)`  
> **Why:** Prevents arbitrary code execution

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
Your deploy script works on your macOS laptop, but fails on a Linux CI runner with `FileNotFoundError` and `UnicodeDecodeError`. Meanwhile, you need to update an AWS IAM JSON policy and verify a release tarball's checksum automatically. This chapter shows how `pathlib`, context managers, and `hashlib` make file automation portable and leak-free.

### Concept: Context Managers (`with`)
**What:** Deterministic resource cleanup block  
**Why:** Prevents `Too many open files` descriptor leaks  
**How:** `__exit__` closes handle even on exceptions  
**When:** Every file open operation

📖 *Book listing:* `service-policy.json` Context Manager Update (pp. 85–89)

**Plain-language analogy:** A `with open(...)` block is like a self-closing refrigerator door—even if you drop the milk (an exception), the door still shuts so nothing spoils (no leaked file handles).

**Walkthrough:** The book opens `service-policy.json` in a `with open(...)` block, parses it via `json.load()`, updates `Statement['Resource']` to `'S3'`, and writes it back cleanly without leaking file descriptors.

### Concept: Hashing vs. Encryption
**What:** `hashlib.sha256` vs. `Fernet` AES  
**Why:** Verify integrity vs. protect secrets at rest  
**How:** Hashes are one-way; Fernet is reversible with key  
**When:** SHA-256 on artifacts; Fernet/`sops` on secrets

📖 *Book listing:* SHA-256 Hashing & `Fernet` Encryption (pp. 100–102)

**Plain-language analogy:** A SHA-256 hash is like a tamper-evident wax seal (you can verify nothing changed, but can't reconstruct the letter), while `Fernet` encryption is a locked safe where only someone with the key can read inside.

**Walkthrough:** The book converts string secrets to `bytes` via `.encode()` before passing them to `hashlib` for one-way digests, then contrasts that with `Fernet` symmetric AES keys for reversible secret storage.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: JSON read/modify/write with pathlib
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: JSON read/modify/write with pathlib
import json, tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    policy_file = Path(tmp) / 'iam_policy.json'
    policy = {'Version': '2026-01-01', 'Statement': [{'Effect': 'Allow', 'Resource': 'EC2'}]}
    policy_file.write_text(json.dumps(policy, indent=2), encoding='utf-8')
    loaded = json.loads(policy_file.read_text(encoding='utf-8'))
    loaded['Statement'][0]['Resource'] = 'S3'
    print('Updated policy:', loaded)

Updated policy: {'Version': '2026-01-01', 'Statement': [{'Effect': 'Allow', 'Resource': 'S3'}]}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: CSV DictReader & XML namespaces
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: CSV DictReader & XML namespaces
import csv, io
import xml.etree.ElementTree as ET

csv_buf = io.StringIO('host,env\nweb-01,prod\nweb-02,staging\n')
print('CSV rows:', list(csv.DictReader(csv_buf)))

xml_payload = '<service xmlns:cfg="http://example.com/cfg"><cfg:port>8443</cfg:port></service>'
root = ET.fromstring(xml_payload)
print('XML port:', root.find('cfg:port', {'cfg': 'http://example.com/cfg'}).text)

CSV rows: [{'host': 'web-01', 'env': 'prod'}, {'host': 'web-02', 'env': 'staging'}]
XML port: 8443


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: SHA-256 hashing with hashlib
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: SHA-256 hashing with hashlib
import hashlib
print('SHA-256:', hashlib.sha256(b'artifact-binary-payload').hexdigest())

SHA-256: d32f5f083e6244e97aa10e18ce4ca27c3f833e0a862f58ab8406af74e9746fff


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Streaming, `os.walk`, YAML & Encryption

#### Example 4: Intermediate Example 1: Lazy line streaming & chunked binary reads
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Lazy line streaming & chunked binary reads
import io, os, tempfile
from pathlib import Path

def stream_errors(file_obj):
    for line in file_obj:
        if 'ERROR' in line:
            yield line.strip()

print('Errors:', list(stream_errors(io.StringIO('INFO ok\nERROR db timeout\n'))))

with tempfile.TemporaryDirectory() as tmp:
    p = Path(tmp) / 'var' / 'log' / 'app.log'
    p.parent.mkdir(parents=True)
    p.write_text('started', encoding='utf-8')
    for dirpath, _, files in os.walk(tmp):
        for name in files:
            fp = Path(dirpath) / name
            print('Walked:', fp.relative_to(tmp), '| bytes:', fp.stat().st_size)

Errors: ['ERROR db timeout']
Walked: var/log/app.log | bytes: 7


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): YAML & Fernet
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): YAML & Fernet
# Requires: pip install pyyaml cryptography
try:
    import yaml
    cfg = yaml.safe_load('service: api\nreplicas: 3')
    print('YAML parsed:', cfg)
except ImportError:
    print('PyYAML not installed; fallback:', {'service': 'api', 'replicas': 3})

try:
    from cryptography.fernet import Fernet
    f = Fernet(Fernet.generate_key())
    tok = f.encrypt(b'secret')
    print('Fernet roundtrip:', f.decrypt(tok))
except ImportError:
    print('cryptography not installed; install via: pip install cryptography')

YAML parsed: {'service': 'api', 'replicas': 3}
cryptography not installed; install via: pip install cryptography


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Atomic Config Writes**—writing to a temp file and calling `Path.replace()` so readers never observe half-written configs.

#### Example 6: import json, tempfile
*Expected output:* Printed labelled output below.

In [6]:
import json, tempfile
from pathlib import Path

def atomic_write_json(target: Path, payload: dict) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp_file = target.with_suffix('.tmp')
    tmp_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    tmp_file.replace(target)

with tempfile.TemporaryDirectory() as tmp:
    p = Path(tmp) / 'cfg.json'
    atomic_write_json(p, {'workers': 4})
    print('Atomic config:', json.loads(p.read_text()))

Atomic config: {'workers': 4}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: sha256sum CLI vs Python hashlib on a temp file
import hashlib, shutil, subprocess, tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as tmp:
    fp = Path(tmp) / 'sample.txt'
    fp.write_text('devops-checksum', encoding='utf-8')
    py_hash = hashlib.sha256(fp.read_bytes()).hexdigest()
    if shutil.which('sha256sum'):
        cli_hash = subprocess.run(['sha256sum', str(fp)], capture_output=True, text=True, check=True).stdout.split()[0]
        assert py_hash == cli_hash
        print('sha256sum matches hashlib:', cli_hash[:16] + '...')
    else:
        print('hashlib:', py_hash[:16] + '...')

sha256sum matches hashlib: 148d470cbe5b37e1...


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **`yaml.load()` without `SafeLoader`** -> Arbitrary code execution risk. ✅ Always use `yaml.safe_load()`.
- ❌ **Omitting `encoding='utf-8'`** -> `UnicodeDecodeError` on Windows/containers. ✅ Pass `encoding='utf-8'` explicitly.
- ❌ **Calling `.read()` on multi-GB logs** -> Container OOM crash. ✅ Iterate `for line in f:` or read 4096-byte chunks.

#### Example 7: import json
*Expected output:* Printed labelled output below.

In [8]:
import json
try:
    json.loads('{bad:')
except json.JSONDecodeError as err:
    print(f'Caught JSONDecodeError at line {err.lineno}: {err.msg}')

Caught JSONDecodeError at line 1: Expecting property name enclosed in double quotes


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `ensure_dir(path: str | Path) -> Path`: Create directory with parents if missing.
2. **(Beginner)** `update_json_key(path: Path, key: str, value) -> dict`: Read, update key, write back.
3. **(Beginner)** `file_sha256(path: Path, chunk_size: int = 4096) -> str`: Hash file in binary chunks.
4. **(Intermediate)** `scan_logs(log_dir: Path) -> dict[str, int]`: Count `'ERROR'` lines per `*.log` file.
5. **(Intermediate)** `count_by_extension(root: Path) -> dict[str, int]`: Count files by suffix.

## 11. 🏆 Practical DevOps Challenge
`build_manifest(root: Path, output_json: Path) -> dict`: Walk `root`, hash every file, write atomic JSON manifest.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement ensure_dir and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement ensure_dir and run to verify!)
def _run_starter_checks():
    assert ensure_dir('/tmp/devops_ch02_test').exists()
from pathlib import Path

def ensure_dir(path: str | Path) -> Path:
    raise NotImplementedError('Implement ensure_dir')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement ensure_dir


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why is `with open(...)` safer than manual `open()`/`close()`?

<details>
<summary>✅ Answer 1</summary>

`with` guarantees the file closes automatically even if an exception occurs.

</details>

**Q2:** When must you use `'rb'`/`'wb'` instead of `'r'`/`'w'`?

<details>
<summary>✅ Answer 2</summary>

For binary files (images, archives, hashes) so Python does not alter newline bytes.

</details>

**Q3:** How does `os.walk()` simplify tree traversal?

<details>
<summary>✅ Answer 3</summary>

It yields `(dirpath, dirnames, filenames)` recursively without manual recursion.

</details>

**Q4:** Why always use `yaml.safe_load()` instead of `yaml.load()`?

<details>
<summary>✅ Answer 4</summary>

`yaml.load()` can execute arbitrary Python objects from untrusted YAML.

</details>

**Q5:** How does SHA-256 differ from Fernet encryption?

<details>
<summary>✅ Answer 5</summary>

SHA-256 is a one-way digest; Fernet is two-way symmetric encryption.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 3**: Use `while chunk := f.read(chunk_size): h.update(chunk)`.
- **Ex 4**: Use `log_dir.rglob('*.log')`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
import hashlib, json, tempfile
from pathlib import Path

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def update_json_key(path: Path, key: str, value) -> dict:
    data = json.loads(path.read_text(encoding='utf-8'))
    data[key] = value
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')
    return data

def file_sha256(path: Path, chunk_size: int = 4096) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def scan_logs(log_dir: Path) -> dict[str, int]:
    return {
        fp.name: sum(1 for line in fp.open('r', encoding='utf-8', errors='replace') if 'ERROR' in line)
        for fp in log_dir.rglob('*.log')
    }

def count_by_extension(root: Path) -> dict[str, int]:
    counts: dict[str, int] = {}
    for fp in root.rglob('*'):
        if fp.is_file():
            ext = fp.suffix or '<no_ext>'
            counts[ext] = counts.get(ext, 0) + 1
    return counts

def build_manifest(root: Path, output_json: Path) -> dict:
    files_info = [
        {'path': fp.relative_to(root).as_posix(), 'size': fp.stat().st_size, 'sha256': file_sha256(fp)}
        for fp in sorted(root.rglob('*')) if fp.is_file() and fp.resolve() != output_json.resolve()
    ]
    manifest = {'file_count': len(files_info), 'files': files_info}
    atomic_write_json(output_json, manifest)
    return manifest

with tempfile.TemporaryDirectory() as tmp:
    d = ensure_dir(Path(tmp) / 'sub')
    cfg = d / 'cfg.json'
    cfg.write_text('{"env": "dev"}', encoding='utf-8')
    assert update_json_key(cfg, 'env', 'prod')['env'] == 'prod'
    (d / 'app.log').write_text('INFO ok\nERROR fail\n', encoding='utf-8')
    assert scan_logs(Path(tmp)) == {'app.log': 1}
    assert count_by_extension(Path(tmp)) == {'.json': 1, '.log': 1}
    m = build_manifest(Path(tmp), Path(tmp) / 'manifest.json')
    assert m['file_count'] == 2
print('All Chapter 2 assertions PASSED!')

All Chapter 2 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `Path.mkdir(parents=True, exist_ok=True)` — Idempotent dir creation (e.g. `Path('logs/app').mkdir(parents=True, exist_ok=True)`)
- `hashlib.sha256(b).hexdigest()` — Artifact integrity digest (e.g. `hashlib.sha256(data).hexdigest()`)
- `os.walk(root)` / `Path.rglob()` — Recursive tree traversal (e.g. `list(Path('.').rglob('*.log'))`)

### ✅ What You Can Now Do
- Read and write JSON, CSV, XML, and YAML (`yaml.safe_load`) portably with `pathlib`.
- Stream large text and binary files in chunks without exhausting RAM.
- Verify artifact integrity with `hashlib.sha256()` and perform atomic config writes.

### ➡️ Next Chapter
Chapter 3 builds on this by combining file automation with `subprocess.run()`, `argparse` CLIs, and dynamic plugin loading.